# Notebook - proyecto de investigación

Notebook académico basado en el artículo **“An Introduction to the Conjugate Gradient Method Without the Agonizing Pain”** de Jonathan Richard Shewchuk (1994).

El objetivo es construir, desde cero y con enfoque reproducible, un estudio completo de:

- generación de matrices **simétricas definidas positivas** (SPD),
- análisis espectral y número de condición,
- implementación de **Steepest Descent (B1)**,
- implementación de **Conjugate Gradients (B2)**,
- comparación de convergencia, costo iterativo y comportamiento frente al condicionamiento.


## 1. Importaciones, estilo y reproducibilidad

Se fija una semilla aleatoria para obtener resultados reproducibles.

In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "font.size": 11,
})

rng = np.random.default_rng(2026)

## 2. Generación de matrices SPD

Se implementan cinco familias de matrices:

1. **Aleatorias SPD**:

$$
A = B^T B + \alpha I
$$

donde $B$ es una matriz aleatoria y $\alpha > 0$ garantiza que la matriz sea definida positiva.

2. **Tridiagonales de diferencias finitas**:

Matrices con estructura:

$$
A =
\begin{pmatrix}
2 & -1 & 0 & \cdots & 0 \\
-1 & 2 & -1 & \cdots & 0 \\
0 & -1 & 2 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & -1 \\
0 & 0 & 0 & -1 & 2
\end{pmatrix}
$$

3. **Diagonalmente dominantes**:

Se genera una matriz simétrica aleatoria y se refuerza la diagonal para asegurar:

$$
|a_{ii}| > \sum_{j \neq i} |a_{ij}|
$$

lo que garantiza que la matriz sea definida positiva.

4. **Hilbert**:

La matriz de Hilbert está definida por:

$$
H_{ij} = \frac{1}{i + j - 1}
$$

Es simétrica definida positiva, pero altamente mal condicionada.

5. **Diagonal con autovalores prefijados**:

Se construye una matriz diagonal

$$
A = \mathrm{diag}(\lambda_1, \ldots, \lambda_n)
$$

permitiendo controlar exactamente el número de condición espectral:

$$
\kappa(A) = \frac{\lambda_{\max}}{\lambda_{\min}}
$$

In [ ]:
def random_spd(n: int, alpha: float = 1e-2, rng: np.random.Generator | None = None) -> np.ndarray:
    """Genera una matriz SPD aleatoria A = B.T @ B + alpha I."""
    rng = rng or np.random.default_rng()
    B = rng.standard_normal((n, n))
    A = B.T @ B + alpha * np.eye(n)
    return 0.5 * (A + A.T)

def tridiagonal_1d_spd(n: int) -> np.ndarray:
    """Matriz tridiagonal SPD típica de diferencias finitas 1D."""
    A = 2.0 * np.eye(n)
    off = -1.0 * np.ones(n - 1)
    A += np.diag(off, 1) + np.diag(off, -1)
    return A

def diagonally_dominant_spd(n: int, rng: np.random.Generator | None = None) -> np.ndarray:
    """Matriz simétrica con dominancia diagonal reforzada para asegurar SPD."""
    rng = rng or np.random.default_rng()
    M = rng.standard_normal((n, n))
    A = 0.5 * (M + M.T)
    diag_boost = np.sum(np.abs(A), axis=1) + 1.0
    A = A + np.diag(diag_boost)
    return 0.5 * (A + A.T)

def hilbert_spd(n: int) -> np.ndarray:
    """Matriz de Hilbert: SPD, simétrica y extremadamente mal condicionada."""
    return la.hilbert(n)

def prescribed_spectrum_spd(n: int, lam_min: float = 1.0, lam_max: float = 1e4) -> np.ndarray:
    """Matriz diagonal SPD con autovalores prefijados."""
    eigs = np.logspace(np.log10(lam_min), np.log10(lam_max), n)
    return np.diag(eigs)

def matrix_family(name: str, n: int, rng: np.random.Generator | None = None) -> np.ndarray:
    """Fábrica unificada de matrices SPD."""
    if name == "random_spd":
        return random_spd(n, alpha=1e-2, rng=rng)
    if name == "tridiagonal":
        return tridiagonal_1d_spd(n)
    if name == "diagonally_dominant":
        return diagonally_dominant_spd(n, rng=rng)
    if name == "hilbert":
        return hilbert_spd(n)
    if name == "prescribed_spectrum":
        return prescribed_spectrum_spd(n, 1.0, 1e4)
    raise ValueError(f"Unknown matrix family: {name}")

## 3. Análisis espectral

Para cada matriz calculamos todos sus autovalores usando `eigh`, y definimos el número de condición espectral como:

$$
\kappa(A) = \frac{\lambda_{\max}}{\lambda_{\min}}
$$

En matrices extremadamente mal condicionadas, la aritmética de punto flotante puede hacer que el mínimo autovalor calculado sea muy pequeño o incluso numéricamente no positivo. En esos casos se marca la condición como $\infty$ o como numéricamente inestable.

In [ ]:
def spectral_properties(A: np.ndarray) -> dict:
    """Devuelve autovalores, número de condición y banderas de estabilidad numérica."""
    eigs = la.eigh(A, eigvals_only=True)
    lam_min = float(np.min(eigs))
    lam_max = float(np.max(eigs))
    stable = bool(lam_min > 0 and np.isfinite(lam_min) and np.isfinite(lam_max))
    kappa = lam_max / lam_min if stable else np.inf
    return {
        "eigenvalues": eigs,
        "lambda_min": lam_min,
        "lambda_max": lam_max,
        "kappa": kappa,
        "stable": stable,
    }

sizes = [10, 50, 100, 200]
families = ["random_spd", "tridiagonal", "diagonally_dominant", "hilbert", "prescribed_spectrum"]

spectral_rows = []
for family in families:
    for n in sizes:
        A = matrix_family(family, n, rng=rng)
        sp = spectral_properties(A)
        spectral_rows.append({
            "family": family,
            "n": n,
            "lambda_min": sp["lambda_min"],
            "lambda_max": sp["lambda_max"],
            "kappa": sp["kappa"],
            "stable": sp["stable"],
            "A": A,
        })

print(f"{'family':<22} {'n':>4} {'lambda_min':>14} {'lambda_max':>14} {'kappa':>16} {'stable':>10}")
print("-" * 82)
for row in spectral_rows:
    kappa_str = f"{row['kappa']:.4e}" if np.isfinite(row["kappa"]) else "inf"
    print(f"{row['family']:<22} {row['n']:>4d} {row['lambda_min']:>14.6e} {row['lambda_max']:>14.6e} {kappa_str:>16} {str(row['stable']):>10}")

## 4. Algoritmo B1: Steepest Descent

La implementación sigue la idea del artículo.

El residual se define como:

$$
r_k = b - A x_k
$$

El tamaño de paso está dado por:

$$
\alpha_k = \frac{r_k^T r_k}{r_k^T A r_k}
$$

y la actualización de la solución es:

$$
x_{k+1} = x_k + \alpha_k r_k.
$$

Se usa la actualización recursiva del residual y, cada cierto número de iteraciones, se recalcula exactamente para evitar deriva numérica.

In [ ]:
def steepest_descent(
    A: np.ndarray,
    b: np.ndarray,
    x0: np.ndarray | None = None,
    tol: float = 1e-10,
    max_iter: int = 5000,
    refresh_every: int = 50,
    x_true: np.ndarray | None = None,
):
    """
    Steepest Descent (B1) desde cero.
    Retorna x, historial y número de iteraciones realizadas.
    """
    n = A.shape[0]
    x = np.zeros(n) if x0 is None else x0.astype(float).copy()
    r = b - A @ x
    rr = float(r @ r)

    history = {
        "residual_norm": [np.sqrt(rr)],
        "error_a_norm": [],
    }

    if x_true is not None:
        e = x - x_true
        history["error_a_norm"].append(np.sqrt(float(e @ (A @ e))))

    it = 0
    while it < max_iter and np.sqrt(rr) > tol:
        Ar = A @ r
        denom = float(r @ Ar)
        if not np.isfinite(denom) or denom <= 0.0:
            r = b - A @ x
            rr = float(r @ r)
            if np.sqrt(rr) <= tol:
                break
            Ar = A @ r
            denom = float(r @ Ar)
            if not np.isfinite(denom) or denom <= 0.0:
                raise FloatingPointError("Steepest Descent encountered a non-positive denominator.")

        alpha = rr / denom
        x = x + alpha * r

        if (it + 1) % refresh_every == 0:
            r = b - A @ x
        else:
            r = r - alpha * Ar

        rr = float(r @ r)
        history["residual_norm"].append(np.sqrt(rr))

        if x_true is not None:
            e = x - x_true
            history["error_a_norm"].append(np.sqrt(float(e @ (A @ e))))

        it += 1

    return x, history, it

## 5. Algoritmo B2: Conjugate Gradients

La versión implementada sigue las fórmulas del artículo para sistemas simétricos definidos positivos (SPD).

Inicialmente se toma la dirección de búsqueda igual al residual:

$$
d_k = r_k
$$

El tamaño de paso se calcula como:

$$
\alpha_k = \frac{r_k^T r_k}{d_k^T A d_k}
$$

y la actualización de la solución es:

$$
x_{k+1} = x_k + \alpha_k d_k.
$$

El residual se actualiza mediante:

$$
r_{k+1} = r_k - \alpha_k A d_k
$$

Luego se calcula el coeficiente de conjugación:

$$
\beta_k = \frac{r_{k+1}^T r_{k+1}}{r_k^T r_k}
$$

y la nueva dirección de búsqueda queda:

$$
d_{k+1} = r_{k+1} + \beta_k d_k.
$$

También se incluye la recomputación exacta del residual cada 50 iteraciones para evitar acumulación de error numérico, tal como recomienda el artículo.

In [ ]:
def conjugate_gradients(
    A: np.ndarray,
    b: np.ndarray,
    x0: np.ndarray | None = None,
    tol: float = 1e-10,
    max_iter: int = 5000,
    refresh_every: int = 50,
    x_true: np.ndarray | None = None,
):
    """
    Conjugate Gradients (B2) desde cero.
    Retorna x, historial y número de iteraciones realizadas.
    """
    n = A.shape[0]
    x = np.zeros(n) if x0 is None else x0.astype(float).copy()
    r = b - A @ x
    d = r.copy()
    rr = float(r @ r)

    history = {
        "residual_norm": [np.sqrt(rr)],
        "error_a_norm": [],
    }

    if x_true is not None:
        e = x - x_true
        history["error_a_norm"].append(np.sqrt(float(e @ (A @ e))))

    it = 0
    while it < max_iter and np.sqrt(rr) > tol:
        Ad = A @ d
        denom = float(d @ Ad)
        if not np.isfinite(denom) or denom <= 0.0:
            r = b - A @ x
            d = r.copy()
            rr = float(r @ r)
            if np.sqrt(rr) <= tol:
                break
            Ad = A @ d
            denom = float(d @ Ad)
            if not np.isfinite(denom) or denom <= 0.0:
                raise FloatingPointError("CG encountered a non-positive denominator.")

        alpha = rr / denom
        x = x + alpha * d

        if (it + 1) % refresh_every == 0:
            r_new = b - A @ x
        else:
            r_new = r - alpha * Ad

        rr_new = float(r_new @ r_new)
        if rr == 0.0:
            break

        beta = rr_new / rr
        d = r_new + beta * d
        r = r_new
        rr = rr_new

        history["residual_norm"].append(np.sqrt(rr))
        if x_true is not None:
            e = x - x_true
            history["error_a_norm"].append(np.sqrt(float(e @ (A @ e))))

        it += 1

    return x, history, it

## 6. Rutina de experimentación

Se construye un vector solución verdadero $x_{\text{true}}$, se define

$$
b = A x_{\text{true}}
$$

y luego se resuelven los sistemas partiendo desde

$$
x_0 = 0.
$$

Durante la ejecución se registran:

- la norma del residual,
- la norma del error en energía

$$
\|e_k\|_A
$$

- el número de iteraciones,
- el error final frente a la solución exacta.

In [ ]:
def run_experiment(A: np.ndarray, label: str, tol: float = 1e-10, max_iter_factor: int = 25):
    n = A.shape[0]
    x_true = rng.standard_normal(n)
    b = A @ x_true
    x0 = np.zeros(n)

    max_iter_sd = max(50, max_iter_factor * n)
    max_iter_cg = max(20, max_iter_factor * n)

    x_sd, hist_sd, it_sd = steepest_descent(
        A, b, x0=x0, tol=tol, max_iter=max_iter_sd, refresh_every=50, x_true=x_true
    )
    x_cg, hist_cg, it_cg = conjugate_gradients(
        A, b, x0=x0, tol=tol, max_iter=max_iter_cg, refresh_every=50, x_true=x_true
    )

    err_sd = np.linalg.norm(x_sd - x_true)
    err_cg = np.linalg.norm(x_cg - x_true)

    return {
        "label": label,
        "A": A,
        "x_true": x_true,
        "b": b,
        "sd": {"x": x_sd, "history": hist_sd, "iters": it_sd, "err_final": err_sd},
        "cg": {"x": x_cg, "history": hist_cg, "iters": it_cg, "err_final": err_cg},
    }

def theory_bounds(kappa: float, e0_a: float, n_steps: int):
    steps = np.arange(n_steps)
    q_sd = (kappa - 1.0) / (kappa + 1.0) if np.isfinite(kappa) and kappa > 1 else 0.0
    q_cg = (np.sqrt(kappa) - 1.0) / (np.sqrt(kappa) + 1.0) if np.isfinite(kappa) and kappa > 1 else 0.0
    sd_bound = e0_a * (q_sd ** steps)
    cg_bound = 2.0 * e0_a * (q_cg ** steps)
    return sd_bound, cg_bound

def print_experiment_summary(result: dict, kappa: float):
    sd = result["sd"]
    cg = result["cg"]
    print(f"\n=== {result['label']} ===")
    print(f"kappa ≈ {kappa:.6e}" if np.isfinite(kappa) else "kappa ≈ inf")
    print(f"Steepest Descent: iters={sd['iters']:>4d}, final ||x-x_true||2={sd['err_final']:.3e}, final residual={sd['history']['residual_norm'][-1]:.3e}")
    print(f"CG:               iters={cg['iters']:>4d}, final ||x-x_true||2={cg['err_final']:.3e}, final residual={cg['history']['residual_norm'][-1]:.3e}")

## 7. Tablas de resultados espectrales

La siguiente celda recorre todas las familias de matrices y tamaños pedidos, y después guarda una tabla de resumen de convergencia para algunos casos representativos.

In [ ]:
print("\nRESUMEN ESPECTRAL")
print("=" * 90)
for family in families:
    print(f"\nFamilia: {family}")
    print(f"{'n':>4} {'lambda_min':>15} {'lambda_max':>15} {'kappa':>16}")
    print("-" * 54)
    for row in spectral_rows:
        if row["family"] != family:
            continue
        kappa_str = f"{row['kappa']:.6e}" if np.isfinite(row["kappa"]) else "inf"
        print(f"{row['n']:>4d} {row['lambda_min']:>15.6e} {row['lambda_max']:>15.6e} {kappa_str:>16}")

## 8. Comparación de convergencia en un caso moderadamente condicionado

Se usa una matriz diagonal con espectro prefijado y

$$
\kappa \approx 50.
$$

Este caso permite ver claramente la diferencia entre Steepest Descent y Conjugate Gradients sin que el problema sea excesivamente difícil.

In [ ]:
n_demo = 50
A_demo = prescribed_spectrum_spd(n_demo, lam_min=1.0, lam_max=50.0)
sp_demo = spectral_properties(A_demo)
result_demo = run_experiment(A_demo, "Diagonal with prescribed spectrum (kappa≈50)", tol=1e-12, max_iter_factor=20)
print_experiment_summary(result_demo, sp_demo["kappa"])

In [ ]:
sd_hist = result_demo["sd"]["history"]
cg_hist = result_demo["cg"]["history"]
e0_a = sd_hist["error_a_norm"][0]
n_steps = max(len(sd_hist["error_a_norm"]), len(cg_hist["error_a_norm"]))
sd_bound, cg_bound = theory_bounds(sp_demo["kappa"], e0_a, n_steps)

plt.figure(figsize=(10, 6))
plt.semilogy(sd_hist["error_a_norm"], label="Steepest Descent: error A-norm")
plt.semilogy(cg_hist["error_a_norm"], label="CG: error A-norm")
plt.semilogy(sd_bound, "--", label="Cota teórica SD")
plt.semilogy(cg_bound, "--", label="Cota teórica CG")
plt.xlabel("Iteración")
plt.ylabel(r"$\|e_k\|_A$")
plt.title("Convergencia comparativa para una matriz con kappa ≈ 50")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.semilogy(sd_hist["residual_norm"], label="Steepest Descent: residual")
plt.semilogy(cg_hist["residual_norm"], label="CG: residual")
plt.xlabel("Iteración")
plt.ylabel(r"$\|r_k\|_2$")
plt.title("Norma del residual para la misma matriz")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Comparación en un caso muy mal condicionado

La matriz de Hilbert es un ejemplo clásico de mala condición numérica.  
Aquí se toma \(n=10\), donde todavía es posible observar el efecto con claridad.

In [ ]:
A_hil = hilbert_spd(10)
sp_hil = spectral_properties(A_hil)
result_hil = run_experiment(A_hil, "Hilbert matrix (n=10)", tol=1e-12, max_iter_factor=50)
print_experiment_summary(result_hil, sp_hil["kappa"])

In [ ]:
sd_hist = result_hil["sd"]["history"]
cg_hist = result_hil["cg"]["history"]
e0_a = sd_hist["error_a_norm"][0]
n_steps = max(len(sd_hist["error_a_norm"]), len(cg_hist["error_a_norm"]))
sd_bound, cg_bound = theory_bounds(sp_hil["kappa"], e0_a, n_steps)

plt.figure(figsize=(10, 6))
plt.semilogy(sd_hist["error_a_norm"], label="Steepest Descent: error A-norm")
plt.semilogy(cg_hist["error_a_norm"], label="CG: error A-norm")
plt.semilogy(sd_bound, "--", label="Cota teórica SD")
plt.semilogy(cg_bound, "--", label="Cota teórica CG")
plt.xlabel("Iteración")
plt.ylabel(r"$\|e_k\|_A$")
plt.title("Convergencia comparativa para Hilbert(n=10)")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.semilogy(sd_hist["residual_norm"], label="Steepest Descent: residual")
plt.semilogy(cg_hist["residual_norm"], label="CG: residual")
plt.xlabel("Iteración")
plt.ylabel(r"$\|r_k\|_2$")
plt.title("Residual en el caso Hilbert(n=10)")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Resumen de iteraciones en función del número de condición

Para ilustrar la relación entre condicionamiento y velocidad de convergencia, se usa una familia de matrices diagonales con espectro controlado.

Así se obtiene una relación limpia entre

$$
\kappa
$$

y el número de iteraciones necesarias para alcanzar la tolerancia establecida.

In [ ]:
kappas = np.logspace(0, 6, 9)  # de 1 a 10^6
n_fixed = 60

sd_iters = []
cg_iters = []

for kappa_target in kappas:
    A = prescribed_spectrum_spd(n_fixed, lam_min=1.0, lam_max=float(kappa_target))
    x_true = rng.standard_normal(n_fixed)
    b = A @ x_true
    x0 = np.zeros(n_fixed)

    x_sd, hist_sd, it_sd = steepest_descent(A, b, x0=x0, tol=1e-10, max_iter=50 * n_fixed, refresh_every=50, x_true=x_true)
    x_cg, hist_cg, it_cg = conjugate_gradients(A, b, x0=x0, tol=1e-10, max_iter=10 * n_fixed, refresh_every=50, x_true=x_true)

    sd_iters.append(it_sd)
    cg_iters.append(it_cg)

plt.figure(figsize=(10, 6))
plt.semilogx(kappas, sd_iters, "o-", label="Steepest Descent")
plt.semilogx(kappas, cg_iters, "o-", label="Conjugate Gradients")
plt.xlabel(r"Condición $\kappa$")
plt.ylabel("Iteraciones hasta tolerancia")
plt.title("Iteraciones vs. número de condición (tamaño fijo)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"{'kappa':>14} {'SD iters':>12} {'CG iters':>12}")
print("-" * 40)
for kappa_target, si, ci in zip(kappas, sd_iters, cg_iters):
    print(f"{kappa_target:>14.4e} {si:>12d} {ci:>12d}")

## 11. Conclusiones

- **Steepest Descent** mejora la solución de forma estable, pero su progreso se vuelve lento cuando el número de condición crece.
- **Conjugate Gradients** explota mejor la estructura espectral de la matriz SPD y, en la práctica, converge con muchas menos iteraciones.
- Las matrices con espectro más compacto, por ejemplo, las tridiagonales bien comportadas tienden a ser más fáciles para ambos métodos.
- Las matrices con espectro muy disperso, como Hilbert, muestran con claridad por qué el condicionamiento domina la dificultad numérica.
- La versión con actualización recursiva del residual es eficiente, pero conviene recomputar el residual exacto ocasionalmente para evitar deriva por redondeo.

